<a href="https://colab.research.google.com/github/zach-yan/ORFE-Network-Project/blob/main/Find_Affliations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import time
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 1. Load the data
df = pd.read_csv("orfe_nodes_with_classifications.csv")

# Identify authors missing affiliations (handling both actual NaNs and "Unknown" strings)
missing_mask = df['Affiliation'].isna() | (df['Affiliation'] == 'Unknown')
missing_ids = df[missing_mask]['Id'].dropna().astype(str).tolist()

print(f"Found {len(missing_ids)} authors missing affiliations. Beginning rescue pass...")

# 2. Define the Targeted API Fetcher
def fetch_missing_affiliations(author_ids, api_key=None, batch_size=50, max_retries=5):
    """Fetches ONLY the affiliations field to quickly fill in missing data."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Accept": "application/json"
    }

    if api_key and api_key != "YOUR_KEY" and str(api_key).strip() != "":
        headers["x-api-key"] = api_key

    base_url = "https://api.semanticscholar.org/graph/v1/author/batch"

    # Pulling affiliations
    query_params = {"fields": "affiliations"}

    recovered_affiliations = {}

    for i in range(0, len(author_ids), batch_size):
        batch = author_ids[i:i+batch_size]
        success = False
        retries = 0
        backoff_time = 2

        while not success and retries < max_retries:
            try:
                response = requests.post(base_url, headers=headers, params=query_params, json={"ids": batch})

                if response.status_code == 200:
                    results = response.json()
                    for author in results:
                        if not author:
                            continue

                        # Semantic Scholar returns affiliations as a list.
                        # We join them with a semicolon if there are multiple.
                        affil_list = author.get('affiliations', [])
                        if affil_list:
                            recovered_affiliations[author['authorId']] = " ; ".join(affil_list)

                    success = True
                    time.sleep(0.5) # Gentle pacing

                elif response.status_code == 429:
                    retries += 1
                    logging.warning(f"Rate limit at batch {i}. Retrying in {backoff_time}s...")
                    time.sleep(backoff_time)
                    backoff_time *= 2

                elif response.status_code == 403:
                    logging.error("403 Forbidden. IP blocked or invalid key. Halting.")
                    return recovered_affiliations # Return what we have so far safely

                elif response.status_code == 400:
                    logging.error(f"400 Bad Request at batch {i}. Malformed ID. Skipping.")
                    break # Skip batch

                else:
                    logging.error(f"Error {response.status_code} at batch {i}. Skipping.")
                    break

            except requests.exceptions.RequestException as e:
                retries += 1
                logging.warning(f"Network error: {e}. Retrying in {backoff_time}s...")
                time.sleep(backoff_time)
                backoff_time *= 2

        # Simple progress tracker
        if (i + batch_size) % 1000 == 0:
            logging.info(f"Processed {i + batch_size} authors... Recovered {len(recovered_affiliations)} affiliations so far.")

    return recovered_affiliations

# 3. Execute the Fetcher
from google.colab import userdata
API_KEY = userdata.get('SemanticScholarAPI')

recovered_dict = fetch_missing_affiliations(missing_ids, api_key=API_KEY)
print(f"\nRescue pass complete! Successfully recovered affiliations for {len(recovered_dict)} authors.")

# 4. Map the recovered data back into the DataFrame
# Convert the keys to strings just in case, for a safe map
recovered_dict_str_keys = {str(k): v for k, v in recovered_dict.items()}

# Apply the mapping only to the rows that were missing affiliations
df.loc[missing_mask, 'Affiliation'] = df.loc[missing_mask, 'Id'].astype(str).map(recovered_dict_str_keys).fillna('Unknown')

# 5. Save the updated network
output_filename = "orfe_nodes_affiliations_rescued.csv"
df.to_csv(output_filename, index=False)
print(f"Saved updated nodes to {output_filename}")

# Take a peek at how many are still unknown vs filled
print("\nCurrent Affiliation Value Counts (Top 10):")
print(df['Affiliation'].value_counts().head(10))

Found 22786 authors missing affiliations. Beginning rescue pass...



Rescue pass complete! Successfully recovered affiliations for 1 authors.
Saved updated nodes to orfe_nodes_affiliations_rescued.csv

Current Affiliation Value Counts (Top 10):
Affiliation
Unknown                                  22785
Stanford University                         20
UC Berkeley                                 15
Carnegie Mellon University                  11
Princeton University                        10
University of Pennsylvania                   8
University of California, Berkeley           8
University of Washington                     8
Massachusetts Institute of Technology        8
Arizona State University                     8
Name: count, dtype: int64


Try finding author affiliations through their papers.



In [ ]:
import pandas as pd
import requests
import time
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 1. Load both datasets
nodes_df = pd.read_csv("orfe_nodes_affiliations_rescued.csv")
papers_df = pd.read_csv("author_papers_final.csv") # The file from your text ingestion!

# 2. Identify the authors who are STILL missing affiliations
missing_mask = nodes_df['Affiliation'] == 'Unknown'
missing_author_ids = set(nodes_df[missing_mask]['Id'].dropna().astype(str))

print(f"Targeting {len(missing_author_ids)} authors for paper-level extraction.")

# 3. Find the papers written by these missing authors
# We filter the papers dataframe to only include rows belonging to the missing authors
targeted_papers_df = papers_df[papers_df['Author_Id'].astype(str).isin(missing_author_ids)]
unique_paper_ids = targeted_papers_df['Paper_Id'].dropna().astype(str).unique().tolist()

print(f"Found {len(unique_paper_ids)} unique recent papers to query for affiliation metadata.")

def fetch_paper_affiliations(paper_ids, target_author_set, api_key=None, batch_size=50):
    """Fetches paper metadata to extract author affiliations at the time of publication."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json"
    }
    if api_key and api_key != "YOUR_KEY":
        headers["x-api-key"] = api_key

    # Notice we are hitting the /paper/batch endpoint now, not the author endpoint!
    base_url = "https://api.semanticscholar.org/graph/v1/paper/batch"

    # We ask for the authors array on the paper, which contains their IDs and affiliations
    query_params = {"fields": "authors.authorId,authors.affiliations"}

    recovered_affiliations = {}

    for i in range(0, len(paper_ids), batch_size):
        batch = paper_ids[i:i+batch_size]
        success = False
        retries = 0
        backoff_time = 2

        while not success and retries < 5:
            try:
                response = requests.post(base_url, headers=headers, params=query_params, json={"ids": batch})

                if response.status_code == 200:
                    results = response.json()
                    for paper in results:
                        if not paper or 'authors' not in paper:
                            continue

                        # Loop through the authors on this specific paper
                        for author in paper['authors']:
                            aid = str(author.get('authorId'))

                            # If this author is one of our missing targets AND they have an affiliation listed
                            if aid in target_author_set and author.get('affiliations'):
                                # If we haven't found an affiliation for them yet, save it
                                if aid not in recovered_affiliations:
                                    # Take the most recent one listed on the paper
                                    recovered_affiliations[aid] = " ; ".join(author['affiliations'])

                    success = True
                    time.sleep(0.5)

                elif response.status_code == 429:
                    retries += 1
                    time.sleep(backoff_time)
                    backoff_time *= 2
                elif response.status_code in [400, 403, 404, 500, 502, 504]:
                    break # Skip bad batches

            except requests.exceptions.RequestException:
                retries += 1
                time.sleep(backoff_time)
                backoff_time *= 2

        if (i + batch_size) % 5000 == 0:
            logging.info(f"Processed {i + batch_size} papers... Recovered {len(recovered_affiliations)} author affiliations so far.")

    return recovered_affiliations

# 4. Run the extraction
from google.colab import userdata
API_KEY = userdata.get('SemanticScholarAPI')
recovered_dict = fetch_paper_affiliations(unique_paper_ids, missing_author_ids, api_key=API_KEY)

print(f"\nPaper-Level Rescue complete! Recovered affiliations for {len(recovered_dict)} authors.")

# 5. Map the newly recovered data back into the main nodes dataframe
df_mapped = nodes_df.copy()
df_mapped['Id_str'] = df_mapped['Id'].astype(str)

# Apply the updates only to the rows we found data for
for aid, affil in recovered_dict.items():
    df_mapped.loc[df_mapped['Id_str'] == aid, 'Affiliation'] = affil

df_mapped.drop(columns=['Id_str'], inplace=True)

# Save the final result
output_filename = "orfe_nodes_final_affiliations.csv"
df_mapped.to_csv(output_filename, index=False)
print(f"Saved to {output_filename}")

print("\nNew Affiliation Value Counts (Top 15):")
print(df_mapped['Affiliation'].value_counts().head(15))

Targeting 22785 authors for paper-level extraction.
Found 58669 unique recent papers to query for affiliation metadata.

Paper-Level Rescue complete! Recovered affiliations for 0 authors.
Saved to orfe_nodes_final_affiliations.csv

New Affiliation Value Counts (Top 15):
Affiliation
Unknown                                       22785
Stanford University                              20
UC Berkeley                                      15
Carnegie Mellon University                       11
Princeton University                             10
University of Pennsylvania                        8
University of California, Berkeley                8
University of Washington                          8
Massachusetts Institute of Technology             8
Arizona State University                          8
Google                                            7
National University of Singapore                  7
Salesforce Research                               7
New York University                      


Network Gravity Approach to infer affiliations.



In [ ]:
import pandas as pd
import networkx as nx
from collections import defaultdict

# 1. Load the data
# Use the file we just tried to rescue, plus your original edges file
nodes_df = pd.read_csv("orfe_nodes_affiliations_rescued.csv")
edges_df = pd.read_csv("orfe_edges_fuzzy_cleaned.csv") # Adjust filename if necessary

print(f"Starting Unknowns: {len(nodes_df[nodes_df['Affiliation'] == 'Unknown'])}")

# 2. Establish the Anchors
# We know Layer 0 is the core faculty. Force their affiliation to Princeton to create a massive ground-truth anchor.
nodes_df.loc[nodes_df['Layer'] == 0.0, 'Affiliation'] = 'Princeton University'

# 3. Build the NetworkX Graph
G = nx.Graph()

# Add nodes with their current affiliations
for _, row in nodes_df.iterrows():
    affil = row['Affiliation']
    # Treat actual NaNs and "Unknown" as None for the algorithm
    if pd.isna(affil) or affil == 'Unknown':
        affil = None
    G.add_node(row['Id'], affiliation=affil, layer=row['Layer'])

# Add edges with their weights (co-authorship strength)
for _, row in edges_df.iterrows():
    if G.has_node(row['Source']) and G.has_node(row['Target']):
        G.add_edge(row['Source'], row['Target'], weight=row['Weight'])

# 4. The Gravity Propagation Algorithm
def propagate_affiliations(graph, target_layer):
    """Infers affiliations based on the weighted pull of known neighbors."""
    updates = {}

    for node, data in graph.nodes(data=True):
        if data['layer'] == target_layer and data['affiliation'] is None:
            # Tally the gravitational pull (edge weights) of known neighbors
            institution_weights = defaultdict(float)

            for neighbor in graph.neighbors(node):
                neighbor_affil = graph.nodes[neighbor]['affiliation']
                if neighbor_affil: # Only pull from neighbors with known affiliations
                    edge_weight = graph[node][neighbor].get('weight', 1.0)
                    # Strip the "(Inferred)" tag if it exists so weights stack correctly
                    clean_affil = neighbor_affil.replace(" (Inferred)", "")
                    institution_weights[clean_affil] += edge_weight

            # If the node has ties to known institutions, assign it to the strongest one
            if institution_weights:
                top_institution = max(institution_weights, key=institution_weights.get)
                # Mark it as inferred so you maintain data transparency for your report
                updates[node] = f"{top_institution} (Inferred)"

    # Apply updates synchronously after the full layer is evaluated
    for node, new_affil in updates.items():
        graph.nodes[node]['affiliation'] = new_affil

    return len(updates)

# 5. Execute the Algorithm Layer by Layer
# We propagate outward from the core to ensure information flows logically
print("\nPropagating Gravity...")
layer_1_updates = propagate_affiliations(G, target_layer=1.0)
print(f"Inferred {layer_1_updates} affiliations for Layer 1.")

layer_2_updates = propagate_affiliations(G, target_layer=2.0)
print(f"Inferred {layer_2_updates} affiliations for Layer 2.")

# 6. Map the inferences back to the DataFrame
inferred_dict = {n: d['affiliation'] for n, d in G.nodes(data=True)}
nodes_df['Affiliation'] = nodes_df['Id'].map(inferred_dict).fillna('Unknown')

# 7. Save and Review
nodes_df.to_csv("orfe_nodes_final_gravity.csv", index=False)

print(f"\nFinal Unknowns: {len(nodes_df[nodes_df['Affiliation'] == 'Unknown'])}")
print("\nFinal Affiliation Value Counts (Top 15):")
print(nodes_df['Affiliation'].value_counts().head(15))

Starting Unknowns: 22785

Propagating Gravity...
Inferred 526 affiliations for Layer 1.
Inferred 22243 affiliations for Layer 2.

Final Unknowns: 0

Final Affiliation Value Counts (Top 15):
Affiliation
Princeton University (Inferred)                              6149
Stanford University (Inferred)                               3817
Hong Kong University of Science and Technology (Inferred)    1469
Google (Inferred)                                            1317
Shanghai University of Finance and Economics (Inferred)      1302
UC Berkeley (Inferred)                                       1124
The Chinese University of Hong Kong, Shenzhen (Inferred)      586
Arizona State University (Inferred)                           447
University of Texas at Austin (Inferred)                      407
ETH Zurich (Inferred)                                         389
Salesforce Research (Inferred)                                316
Hospital Universitario 12 de Octubre (Inferred)               287
Massac